In [1]:
import os
import itertools
import multiprocessing as mp

import numpy as np
import pandas as pd
import supervision as sv

from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict

from trackers import ByteTrackTracker
from trackers.eval import evaluate_mot_sequences


# Shared detection utilities (same logic as in the OC-SORT notebook)

def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame[frame_id]:
        det = line.split(",")  # "frame_id,-1,left,top,width,height,conf,-1,-1,-1"
        x1 = int(det[2])
        y1 = int(det[3])
        x2 = int(det[4]) + int(det[2])
        y2 = int(det[5]) + int(det[3])
        conf = 1.0
        dets.append([x1, y1, x2, y2, conf])
    return dets

In [ ]:
def run_bytetrack_with_params(
    lost_track_buffer: int,
    track_activation_threshold: float,
    minimum_consecutive_frames: int,
    minimum_iou_threshold: float,
    high_conf_det_threshold: float,
    eval_set: str = "test",
    use_defaults: bool = False,
):
    """Run ByteTrack on all SoccerNet sequences for a given hyperparameter set."""

    if use_defaults:
        tracker = ByteTrackTracker()
    else:
        tracker = ByteTrackTracker(
            lost_track_buffer=lost_track_buffer,
            track_activation_threshold=track_activation_threshold,
            minimum_consecutive_frames=minimum_consecutive_frames,
            minimum_iou_threshold=minimum_iou_threshold,
            high_conf_det_threshold=high_conf_det_threshold,
        )

    outputs_root = "BYTETRACK_outputs_soccernet"
    os.makedirs(outputs_root, exist_ok=True)
    if use_defaults:
        save_dir = os.path.join(outputs_root, f"{eval_set}_defaults")
    else:
        save_dir = os.path.join(
            outputs_root,
            f"{eval_set}_L{lost_track_buffer}_"
            f"A{track_activation_threshold}_C{minimum_consecutive_frames}_I{minimum_iou_threshold}_H{high_conf_det_threshold}",
        )
    os.makedirs(save_dir, exist_ok=True)

    # Choose detections and GT depending on split
    if eval_set == "train":
        det_root = "SoccerNet_dets/SoccerNet_tracking/train"
        gt_root = "TrackEval/data/gt/SoccerNet_tracking/train"
    elif eval_set == "test":
        # For test we keep using the official all_dets/all_gts folders
        det_root = "SoccerNet_dets/SoccerNet_tracking_2022_all_dets"
        gt_root = "TrackEval/data/gt/SoccerNet_tracking/SoccerNet_tracking_2022_all_gts"
    else:
        raise ValueError(f"Unsupported eval_set: {eval_set}")

    for seq in sorted(os.listdir(det_root)):
        tracker.reset()
        seq_name = seq.split("__")[0]

        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)

        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []

        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(
                    xyxy=raw_dets[:, :4],
                    confidence=raw_dets[:, 4],
                )
            else:
                dets = sv.Detections.empty()

            dets = tracker.update(detections=dets)

            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue

                width = right - left
                height = bottom - top

                output_lines.append(
                    f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                )

        save_path = os.path.join(save_dir, seq_name + ".txt")
        with open(save_path, "w") as f:
            f.writelines(output_lines)

    result = evaluate_mot_sequences(
        gt_dir=gt_root,
        tracker_dir=save_dir,
        metrics=["CLEAR", "HOTA", "Identity"],
    )

    MOTA = result.to_dict()["aggregate"]["CLEAR"]["MOTA"]
    HOTA = result.to_dict()["aggregate"]["HOTA"]["HOTA"]
    IDF1 = result.to_dict()["aggregate"]["Identity"]["IDF1"]
    if use_defaults:
        print(f"ByteTrack | eval_set:{eval_set} | defaults -> HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f}")
    else:
        print(
            f"ByteTrack | eval_set:{eval_set} |  L:{lost_track_buffer}, A:{track_activation_threshold}, "
            f"C:{minimum_consecutive_frames}, I:{minimum_iou_threshold}, H:{high_conf_det_threshold} -> "
            f"HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f}"
        )

    return {
        "lost_track_buffer": lost_track_buffer,
        "track_activation_threshold": track_activation_threshold,
        "minimum_consecutive_frames": minimum_consecutive_frames,
        "minimum_iou_threshold": minimum_iou_threshold,
        "high_conf_det_threshold": high_conf_det_threshold,
        "HOTA": HOTA,
        "IDF1": IDF1,
        "MOTA": MOTA,
    }


In [10]:
# Run ByteTrack with default parameters (ByteTrackTracker()) on the definitive eval set (test)
run_bytetrack_with_params(0, 0.0, 0, 0.0, 0.0, eval_set="test", use_defaults=True)

ByteTrack | eval_set:test | defaults -> HOTA: 0.840, IDF1: 0.781, MOTA: 0.978


{'lost_track_buffer': 0,
 'track_activation_threshold': 0.0,
 'minimum_consecutive_frames': 0,
 'minimum_iou_threshold': 0.0,
 'high_conf_det_threshold': 0.0,
 'HOTA': 0.839888745476606,
 'IDF1': 0.7812787991196446,
 'MOTA': 0.9783738112150095}

In [ ]:
# Define ByteTrack hyperparameter search space

lost_track_buffer_candidates = [10, 30, 60, 90]
track_activation_threshold_candidates = [0.2, 0.5, 0.7, 0.9]
minimum_consecutive_frames_candidates = [1, 3]
minimum_iou_threshold_candidates = [0.05, 0.1, 0.3]
high_conf_det_threshold_candidates = [0.5, 0.6, 0.7, 0.9]

all_candidate_lists = [
    lost_track_buffer_candidates,
    track_activation_threshold_candidates,
    minimum_consecutive_frames_candidates,
    minimum_iou_threshold_candidates,
    high_conf_det_threshold_candidates,
]

parameter_combinations = list(itertools.product(*all_candidate_lists))

print(f"Total ByteTrack parameter combinations: {len(parameter_combinations)}")

Total ByteTrack parameter combinations: 384


In [ ]:
def run_bytetrack_tuning_parallel(parameter_combinations, eval_set: str = "test"):
    num_workers = os.cpu_count()
    print(f"Running {len(parameter_combinations)} ByteTrack combinations with {num_workers} workers")

    ctx = mp.get_context("fork")
    all_results = []

    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        futures = []
        for i, comb in enumerate(parameter_combinations):
            print(f"Submitting combination {i + 1}/{len(parameter_combinations)}")
            (
                lost_track_buffer,
                track_activation_threshold,
                minimum_consecutive_frames,
                minimum_iou_threshold,
                high_conf_det_threshold,
            ) = comb
            futures.append(
                executor.submit(
                    run_bytetrack_with_params,
                    lost_track_buffer,
                    track_activation_threshold,
                    minimum_consecutive_frames,
                    minimum_iou_threshold,
                    high_conf_det_threshold,
                    eval_set,
                )
            )

        for i, f in enumerate(futures):
            try:
                result = f.result()
                all_results.append(result)
                print(f"[{i + 1}/{len(parameter_combinations)}] combination finished.")
            except Exception as e:
                print(f"[{i + 1}/{len(parameter_combinations)}] combination FAILED: {e}")

    df = pd.DataFrame(all_results)
    print("ByteTrack hyperparameter tuning complete. Results stored in 'bytetrack_tuning_df'.")
    print(df.head())

    df.to_csv("bytetrack_soccernet_tuning.csv", index=False)
    return df


bytetrack_tuning_df = run_bytetrack_tuning_parallel(parameter_combinations, eval_set="train")

best_idx = bytetrack_tuning_df["HOTA"].idxmax()
best_row = bytetrack_tuning_df.loc[best_idx]
print("\nBest ByteTrack HOTA row (train):\n", best_row)

best_params = dict(
    lost_track_buffer=int(best_row["lost_track_buffer"]),
    track_activation_threshold=float(best_row["track_activation_threshold"]),
    minimum_consecutive_frames=int(best_row["minimum_consecutive_frames"]),
    minimum_iou_threshold=float(best_row["minimum_iou_threshold"]),
    high_conf_det_threshold=float(best_row["high_conf_det_threshold"]),
)
print("Best params:", best_params)


Running 384 ByteTrack combinations with 10 workers
Submitting combination 1/384
Submitting combination 2/384
Submitting combination 3/384
Submitting combination 4/384
Submitting combination 5/384
Submitting combination 6/384
Submitting combination 7/384
Submitting combination 8/384
Submitting combination 9/384
Submitting combination 10/384
Submitting combination 11/384
Submitting combination 12/384
Submitting combination 13/384
Submitting combination 14/384
Submitting combination 15/384
Submitting combination 16/384
Submitting combination 17/384
Submitting combination 18/384
Submitting combination 19/384
Submitting combination 20/384
Submitting combination 21/384
Submitting combination 22/384
Submitting combination 23/384
Submitting combination 24/384
Submitting combination 25/384
Submitting combination 26/384
Submitting combination 27/384
Submitting combination 28/384
Submitting combination 29/384
Submitting combination 30/384
Submitting combination 31/384
Submitting combination 32/38

KeyboardInterrupt: 

ByteTrack | L:10, A:0.2, C:3, I:0.3, H:0.9 -> HOTA: 0.839, IDF1: 0.789, MOTA: 0.962
ByteTrack | L:10, A:0.2, C:3, I:0.3, H:0.6 -> HOTA: 0.839, IDF1: 0.789, MOTA: 0.962
ByteTrack | L:10, A:0.2, C:3, I:0.3, H:0.5 -> HOTA: 0.839, IDF1: 0.789, MOTA: 0.962
ByteTrack | L:10, A:0.2, C:3, I:0.3, H:0.7 -> HOTA: 0.839, IDF1: 0.789, MOTA: 0.962ByteTrack | L:10, A:0.5, C:1, I:0.05, H:0.7 -> HOTA: 0.860, IDF1: 0.807, MOTA: 0.986

ByteTrack | L:10, A:0.5, C:1, I:0.1, H:0.5 -> HOTA: 0.858, IDF1: 0.806, MOTA: 0.982
ByteTrack | L:10, A:0.5, C:1, I:0.05, H:0.9 -> HOTA: 0.860, IDF1: 0.807, MOTA: 0.986
ByteTrack | L:10, A:0.5, C:1, I:0.05, H:0.6 -> HOTA: 0.860, IDF1: 0.807, MOTA: 0.986
ByteTrack | L:10, A:0.5, C:1, I:0.05, H:0.5 -> HOTA: 0.860, IDF1: 0.807, MOTA: 0.986
ByteTrack | L:10, A:0.5, C:1, I:0.1, H:0.6 -> HOTA: 0.858, IDF1: 0.806, MOTA: 0.982
ByteTrack | L:10, A:0.5, C:1, I:0.1, H:0.9 -> HOTA: 0.858, IDF1: 0.806, MOTA: 0.982
ByteTrack | L:10, A:0.5, C:3, I:0.05, H:0.5 -> HOTA: 0.858, IDF1: 0.808,

In [ ]:

if "best_params" not in globals() or best_params is None:
    raise RuntimeError("best_params is not defined. Run the tuning cell first.")

print("Running ByteTrack on SoccerNet test with:", best_params)
test_result = run_bytetrack_with_params(
    lost_track_buffer=best_params["lost_track_buffer"],
    track_activation_threshold=best_params["track_activation_threshold"],
    minimum_consecutive_frames=best_params["minimum_consecutive_frames"],
    minimum_iou_threshold=best_params["minimum_iou_threshold"],
    high_conf_det_threshold=best_params["high_conf_det_threshold"],
    eval_set="test",
)


Running ByteTrack on SoccerNet test with: {'lost_track_buffer': 30, 'track_activation_threshold': 0.2, 'minimum_consecutive_frames': 1, 'minimum_iou_threshold': 0.05, 'high_conf_det_threshold': 0.5}
ByteTrack | L:30, A:0.2, C:1, I:0.05, H:0.5 -> HOTA: 0.841, IDF1: 0.782, MOTA: 0.982


In [4]:
import pandas as pd

# Load validation tuning results (must include HOTA/IDF1/MOTA columns)
df = pd.read_csv("bytetrack_soccernet_tuning_develop_05_03.csv")

if "HOTA" not in df.columns or df["HOTA"].isna().all():
    raise ValueError(
        "No HOTA values found in 'ocsort_sportsmot_val_tuning.csv'. "
        "Make sure GT is available under SPORTSMOT_GT_ROOT and rerun the tuning cell."
    )

# Row with best HOTA on validation
best_idx = df["HOTA"].idxmax()
best_params = df.loc[best_idx]
print(best_params)
test_result = run_bytetrack_with_params(
    lost_track_buffer=best_params["lost_track_buffer"],
    track_activation_threshold=best_params["track_activation_threshold"],
    minimum_consecutive_frames=best_params["minimum_consecutive_frames"],
    minimum_iou_threshold=best_params["minimum_iou_threshold"],
    high_conf_det_threshold=best_params["high_conf_det_threshold"],
    eval_set="test",
)

lost_track_buffer             30.000000
frame_rate                    25.000000
track_activation_threshold     0.500000
minimum_consecutive_frames     2.000000
minimum_iou_threshold          0.100000
high_conf_det_threshold        0.500000
HOTA                           0.859012
IDF1                           0.806604
MOTA                           0.982480
Name: 72, dtype: float64
ByteTrack | L:30.0, A:0.5, C:2.0, I:0.1, H:0.5 -> HOTA: 0.840, IDF1: 0.781, MOTA: 0.978


In [ ]:
import pandas as pd

# Load validation tuning results (must include HOTA/IDF1/MOTA columns)
df = pd.read_csv("bytetrack_soccernet_tuning.csv")

if "HOTA" not in df.columns or df["HOTA"].isna().all():
    raise ValueError(
        "No HOTA values found in 'ocsort_sportsmot_val_tuning.csv'. "
        "Make sure GT is available under SPORTSMOT_GT_ROOT and rerun the tuning cell."
    )

# Row with best HOTA on validation
best_idx = df["HOTA"].idxmax()
best_params = df.loc[best_idx]
print(best_params)
test_result = run_bytetrack_with_params(
    lost_track_buffer=best_params["lost_track_buffer"],
    track_activation_threshold=best_params["track_activation_threshold"],
    minimum_consecutive_frames=best_params["minimum_consecutive_frames"],
    minimum_iou_threshold=best_params["minimum_iou_threshold"],
    high_conf_det_threshold=best_params["high_conf_det_threshold"],
    eval_set="test",
)

lost_track_buffer             30.000000
track_activation_threshold     0.200000
minimum_consecutive_frames     1.000000
minimum_iou_threshold          0.050000
high_conf_det_threshold        0.500000
HOTA                           0.860293
IDF1                           0.807476
MOTA                           0.985508
Name: 96, dtype: float64


KeyboardInterrupt: 